# Predicting transcription factor family scores from the ATAC data

In [13]:
suppressPackageStartupMessages({
    library(ArchR)
    library(BSgenome.Hsapiens.UCSC.hg38)
    library(Matrix)
    library(GSVA)
    library(fastcluster)
    library(mclust)
    library(pbapply)#
    library(ggplot2)#
    library(dplyr) #
    library(ComplexHeatmap)#
    library(reshape2)#
    library(reticulate)#
    library(data.table)#
    library(stringr)
})

Warning message:
“package ‘GSVA’ was built under R version 4.3.3”
Warning message:
“package ‘fastcluster’ was built under R version 4.3.3”


## Load ArchR project

In [2]:
# Fix random number generator to that used locally 
RNGkind("L'Ecuyer-CMRG")

# Set thread count
addArchRThreads(threads = 1)

# Load ArchR project

proj_fil <- loadArchRProject(path = "../../Data/ArchR_Projects/Save_SW620_DATAC_Results",
                         force = FALSE)


Setting default number of Parallel threads to 1.

Successfully loaded ArchRProject!


                                                   / |
                                                 /    \
            .                                  /      |.
            \\\                              /        |.
              \\\                          /           `|.
                \\\                      /              |.
                  \                    /                |\
                  \\#####\           /                  ||
                ==###########>      /                   ||
                 \\##==......\    /                     ||
            ______ =       =|__ /__                     ||      \\\
        ,--' ,----`-,__ ___/'  --,-`-===================##========>
       \               '        ##_______ _____ ,--,__,=##,__   ///
        ,    __==    ___,-,__,--'#'  ==='      `-'    | ##,-/
        -,____,---'       \\####\\________________,--\\_##,/
        

## Call peaks and get TFBS z-scores

In [4]:
# Add group coverages for peak calling
proj_fil <- addGroupCoverages(ArchRProj = proj_fil, groupBy = "Clusters")

# Set the path to Macs2
pathToMacs2 <- findMacs2()

# Call peaks using Macs2
proj_fil <- addReproduciblePeakSet(
  ArchRProj = proj_fil, 
  groupBy = "Clusters", 
  pathToMacs2 = pathToMacs2
)

# Retrieve a GRanges object containing the peak information (position, signal, etc.)
peaks <- getPeakSet(proj_fil)

# Add peak matrix to the proj_filect
proj_fil <- addPeakMatrix(proj_fil)

# Add TF motif annotations to the ArchR proj_filect
proj_fil <- addMotifAnnotations(ArchRProj = proj_fil, 
                                motifSet = "cisbp",
                                annoName = "Motif_cisbp",
                                force = T)

# Add a set of background peaks to be used for bias-correction of the per cell TF deviation scores
proj_fil <- addBgdPeaks(proj_fil)

ArchR logging to : ArchRLogs/ArchR-addGroupCoverages-c2952d38871d-Date-2026-02-24_Time-14-40-09.037113.log
If there is an issue, please report to github with logFile!

C1 (1 of 3) : CellGroups N = 3

C2 (2 of 3) : CellGroups N = 3

C3 (3 of 3) : CellGroups N = 2

2026-02-24 14:40:10.90608 : Creating Coverage Files!, 0.03 mins elapsed.

2026-02-24 14:40:10.907881 : Batch Execution w/ safelapply!, 0.03 mins elapsed.

2026-02-24 14:40:10.945959 : Group C1._.SW620bc_POTa_DATAC (1 of 8) : Creating Group Coverage File : C1._.SW620bc_POTa_DATAC.insertions.coverage.h5, 0.031 mins elapsed.

Number of Cells = 100

Coverage File Exists!

Added Coverage Group

Added Metadata Group

Added ArrowCoverage Class

Added Coverage/Info

Added Coverage/Info/CellNames

2026-02-24 14:40:28.470113 : Group C1._.SW620bc_POTb_DATAC (2 of 8) : Creating Group Coverage File : C1._.SW620bc_POTb_DATAC.insertions.coverage.h5, 0.323 mins elapsed.

Number of Cells = 68

Coverage File Exists!

Added Coverage Group

Added

   Group nCells nCellsUsed nReplicates nMin nMax maxPeaks
C1    C1    223        223           3   55  100   111500
C2    C2    258        258           3   47  122   129000
C3    C3    187        187           2   64  123    93500


2026-02-24 14:44:50.044993 : Batching Peak Calls!, 0.011 mins elapsed.

2026-02-24 14:44:50.233989 : Batch Execution w/ safelapply!, 0 mins elapsed.

2026-02-24 14:44:50.241222 : Group 1 of 8, Calling Peaks with MACS2!, 0 mins elapsed.

Running Macs2 with Params : macs2 callpeak -g 2.7e+09 --name C1._.SW620bc_POTa_DATAC-1 --treatment /Users/ccolson/Documents/LongTermResistance_CellLineWork/dATACManuscript_AssociatedCode_Revision/Data/ArchR_Projects/Save_SW620_DATAC_Results/PeakCalls/Clusters/InsertionBeds/C1._.SW620bc_POTa_DATAC-1.insertions.bed --outdir /Users/ccolson/Documents/LongTermResistance_CellLineWork/dATACManuscript_AssociatedCode_Revision/Data/ArchR_Projects/Save_SW620_DATAC_Results/PeakCalls/Clusters/InsertionBeds --format BED --call-summits --keep-dup all --nomodel --nolambda --shift -75 --extsize 150 -q 0.1

2026-02-24 14:45:14.537606 : Group 2 of 8, Calling Peaks with MACS2!, 0.405 mins elapsed.

Running Macs2 with Params : macs2 callpeak -g 2.7e+09 --name C1._.SW620bc_P

[1] "/Users/ccolson/Documents/LongTermResistance_CellLineWork/dATACManuscript_AssociatedCode_Revision/Data/ArchR_Projects/Save_SW620_DATAC_Results/PeakCalls/Clusters/C1-reproduciblePeaks.gr.rds"


Annotating Peaks : Nearest Gene

Annotating Peaks : Gene

Annotating Peaks : TSS

Annotating Peaks : GC



[1] "/Users/ccolson/Documents/LongTermResistance_CellLineWork/dATACManuscript_AssociatedCode_Revision/Data/ArchR_Projects/Save_SW620_DATAC_Results/PeakCalls/Clusters/C2-reproduciblePeaks.gr.rds"


Annotating Peaks : Nearest Gene

Annotating Peaks : Gene

Annotating Peaks : TSS

Annotating Peaks : GC



[1] "/Users/ccolson/Documents/LongTermResistance_CellLineWork/dATACManuscript_AssociatedCode_Revision/Data/ArchR_Projects/Save_SW620_DATAC_Results/PeakCalls/Clusters/C3-reproduciblePeaks.gr.rds"


2026-02-24 14:49:07.648576 : Creating Union Peak Set!, 4.305 mins elapsed.

Converged after 5 iterations!

Warning message:
“The `size` argument of `element_line()` is deprecated as of ggplot2 3.4.0.
ℹ Please use the `linewidth` argument instead.
ℹ The deprecated feature was likely used in the ArchR package.
  Please report the issue to the authors.”
Warning message:
“The `size` argument of `element_rect()` is deprecated as of ggplot2 3.4.0.
ℹ Please use the `linewidth` argument instead.
ℹ The deprecated feature was likely used in the ArchR package.
  Please report the issue to the authors.”
Plotting Ggplot!

2026-02-24 14:49:11.194437 : Finished Creating Union Peak Set (132592)!, 4.364 mins elapsed.

ArchR logging to : ArchRLogs/ArchR-addPeakMatrix-c29567984ccc-Date-2026-02-24_Time-14-49-11.216372.log
If there is an issue, please report to github with logFile!

2026-02-24 14:49:11.347719 : Batch Execution w/ safelapply!, 0 mins elapsed.

2026-02-24 14:49:11.782224 : Adding SW620bc_DT1

In [9]:
saveArchRProject(ArchRProj = proj_fil, outputDirectory = "../../Data/ArchR_Projects/Save_SW620_DATAC_Results", load = FALSE)

Saving ArchRProject...



In [3]:
# Add a deviation matrix to the ArchR proj_filect
proj_fil <- addDeviationsMatrix(
  ArchRProj = proj_fil, 
  peakAnnotation = "Motif_cisbp",
  force = TRUE
)

Using Previous Background Peaks!

ArchR logging to : ArchRLogs/ArchR-addDeviationsMatrix-c9193ed99654-Date-2026-02-24_Time-15-26-54.897088.log
If there is an issue, please report to github with logFile!

2026-02-24 15:26:57.887836 : Batch Execution w/ safelapply!, 0 mins elapsed.

Can not create group. Object with name 'Motif_cisbpMatrix' already exists.

2026-02-24 15:26:58.328281 : chromVAR deviations SW620bc_DT1_P1a_DATAC (1 of 8) Schep (2017), 0.007 mins elapsed.



SW620bc_DT1_P1a_DATAC (1 of 8) : Deviations for Annotation 87 of 869
SW620bc_DT1_P1a_DATAC (1 of 8) : Deviations for Annotation 173 of 869
SW620bc_DT1_P1a_DATAC (1 of 8) : Deviations for Annotation 259 of 869
SW620bc_DT1_P1a_DATAC (1 of 8) : Deviations for Annotation 345 of 869
SW620bc_DT1_P1a_DATAC (1 of 8) : Deviations for Annotation 431 of 869
SW620bc_DT1_P1a_DATAC (1 of 8) : Deviations for Annotation 517 of 869
SW620bc_DT1_P1a_DATAC (1 of 8) : Deviations for Annotation 603 of 869
SW620bc_DT1_P1a_DATAC (1 of 8) : Deviations for Annotation 689 of 869
SW620bc_DT1_P1a_DATAC (1 of 8) : Deviations for Annotation 775 of 869
SW620bc_DT1_P1a_DATAC (1 of 8) : Deviations for Annotation 861 of 869


2026-02-24 15:28:59.064524 : Finished Computing Deviations!, 2.02 mins elapsed.

2026-02-24 15:28:59.474057 : chromVAR deviations SW620bc_DT1_P1b_DATAC (2 of 8) Schep (2017), 2.026 mins elapsed.



SW620bc_DT1_P1b_DATAC (2 of 8) : Deviations for Annotation 87 of 869
SW620bc_DT1_P1b_DATAC (2 of 8) : Deviations for Annotation 173 of 869
SW620bc_DT1_P1b_DATAC (2 of 8) : Deviations for Annotation 259 of 869
SW620bc_DT1_P1b_DATAC (2 of 8) : Deviations for Annotation 345 of 869
SW620bc_DT1_P1b_DATAC (2 of 8) : Deviations for Annotation 431 of 869
SW620bc_DT1_P1b_DATAC (2 of 8) : Deviations for Annotation 517 of 869
SW620bc_DT1_P1b_DATAC (2 of 8) : Deviations for Annotation 603 of 869
SW620bc_DT1_P1b_DATAC (2 of 8) : Deviations for Annotation 689 of 869
SW620bc_DT1_P1b_DATAC (2 of 8) : Deviations for Annotation 775 of 869
SW620bc_DT1_P1b_DATAC (2 of 8) : Deviations for Annotation 861 of 869


2026-02-24 15:30:24.444703 : Finished Computing Deviations!, 3.443 mins elapsed.

2026-02-24 15:30:24.853245 : chromVAR deviations SW620bc_DSIa_DATAC (3 of 8) Schep (2017), 3.449 mins elapsed.



...

2026-02-24 15:34:25.820409 : Finished Computing Deviations!, 7.466 mins elapsed.

2026-02-24 15:34:26.229091 : chromVAR deviations SW620bc_DSIVb_DATAC (7 of 8) Schep (2017), 7.472 mins elapsed.



SW620bc_DSIVb_DATAC (7 of 8) : Deviations for Annotation 87 of 869
SW620bc_DSIVb_DATAC (7 of 8) : Deviations for Annotation 173 of 869
SW620bc_DSIVb_DATAC (7 of 8) : Deviations for Annotation 259 of 869
SW620bc_DSIVb_DATAC (7 of 8) : Deviations for Annotation 345 of 869
SW620bc_DSIVb_DATAC (7 of 8) : Deviations for Annotation 431 of 869
SW620bc_DSIVb_DATAC (7 of 8) : Deviations for Annotation 517 of 869
SW620bc_DSIVb_DATAC (7 of 8) : Deviations for Annotation 603 of 869
SW620bc_DSIVb_DATAC (7 of 8) : Deviations for Annotation 689 of 869
SW620bc_DSIVb_DATAC (7 of 8) : Deviations for Annotation 775 of 869
SW620bc_DSIVb_DATAC (7 of 8) : Deviations for Annotation 861 of 869


2026-02-24 15:35:17.957369 : Finished Computing Deviations!, 8.334 mins elapsed.

2026-02-24 15:35:18.363813 : chromVAR deviations SW620bc_DSIVa_DATAC (8 of 8) Schep (2017), 8.341 mins elapsed.



SW620bc_DSIVa_DATAC (8 of 8) : Deviations for Annotation 87 of 869
SW620bc_DSIVa_DATAC (8 of 8) : Deviations for Annotation 173 of 869
SW620bc_DSIVa_DATAC (8 of 8) : Deviations for Annotation 259 of 869
SW620bc_DSIVa_DATAC (8 of 8) : Deviations for Annotation 345 of 869
SW620bc_DSIVa_DATAC (8 of 8) : Deviations for Annotation 431 of 869
SW620bc_DSIVa_DATAC (8 of 8) : Deviations for Annotation 517 of 869
SW620bc_DSIVa_DATAC (8 of 8) : Deviations for Annotation 603 of 869
SW620bc_DSIVa_DATAC (8 of 8) : Deviations for Annotation 689 of 869
SW620bc_DSIVa_DATAC (8 of 8) : Deviations for Annotation 775 of 869
SW620bc_DSIVa_DATAC (8 of 8) : Deviations for Annotation 861 of 869


2026-02-24 15:36:07.589306 : Finished Computing Deviations!, 9.162 mins elapsed.

###########
2026-02-24 15:36:07.591167 : Completed Computing Deviations!, 9.211 mins elapsed.
###########

ArchR logging successful to : ArchRLogs/ArchR-addDeviationsMatrix-c9193ed99654-Date-2026-02-24_Time-15-26-54.897088.log



In [5]:
# Get the TF enrichment matrices from Arch R
motifs <- getMatrixFromProject(proj_fil, useMatrix = "Motif_cisbpMatrix")

# Extract the z-score matrix
motif_mat_z <-  as.matrix(motifs@assays@data@listData$z)

# Edit the cell ids to match the CNA data
write_csv(data.frame("Cell_id" = colnames(motif_mat_z),t(motif_mat_z)), "../../Data/ATAC_Data/TFBSScores/TFBS_Motif_Matrix.csv")

ArchR logging to : ArchRLogs/ArchR-getMatrixFromProject-c91927bbf022-Date-2026-02-24_Time-15-47-20.89684.log
If there is an issue, please report to github with logFile!

2026-02-24 15:47:24.688695 : Organizing colData, 0.063 mins elapsed.

2026-02-24 15:47:24.780129 : Organizing rowData, 0.065 mins elapsed.

2026-02-24 15:47:24.785647 : Organizing rowRanges, 0.065 mins elapsed.

2026-02-24 15:47:24.794633 : Organizing Assays (1 of 2), 0.065 mins elapsed.

2026-02-24 15:47:24.811155 : Organizing Assays (2 of 2), 0.065 mins elapsed.

2026-02-24 15:47:24.828051 : Constructing SummarizedExperiment, 0.066 mins elapsed.

2026-02-24 15:47:25.203288 : Finished Matrix Creation, 0.072 mins elapsed.



In [7]:
# Save the project
saveArchRProject(ArchRProj = proj_fil, outputDirectory = "../../Data/ArchR_Projects/Save_SW620_DATAC_Results", load = FALSE)

Saving ArchRProject...



## Load the TF families

In [11]:
# Read in the TFBS families
families <- read_csv("../../Data/ATAC_Data/TFBSScores/TF_Families.csv", col_types = "ccc")

# Create a family RDS
features <- list()
tfbs_families <- unique(families$Family)
for (i in (tfbs_families)){
  features <- append(features, list(c(families[families$Family == i,]$TF)))
}

names(features) <- tfbs_families

saveRDS(features, "../../Data/ATAC_Data/TFBSScores/TFBS_Families.rds")

## Bootstrapping procedure to impute TFBS activity scores using MAGIC, compute TF family binding site activity scores and cluster cells into TF classes

In [ ]:
# Ensure MAGIC is loaded
# Import magic in python
import("magic")
# Import magic R library
library(Rmagic)

# ---------------- Defined user parameters ----------------
input_csv       <- "../../Data/ATAC_Data/TFBSScores/TFBS_Motif_Matrix.csv"    # rows=cells, cols=genes
families_rds    <- "../../Data/ATAC_Data/TFBSScores/TFBS_Families.rds"      # optional RDS containing named list of gene sets
n_clusters      <- 4  # number of clusters for cutree
n_boot          <- 1000    # number of bootstraps
avg_duplicates  <- TRUE      # average duplicate cells in bootstraps
seed_base       <- 7
save_prefix     <- "TFBS_dATACOnly_Jan26/magic_gsva_bootstrap_zscoreParam"
n_cores         <- 1         # parallel workers for GSVA

set.seed(seed_base)

# ---------------- Load data ----------------
cat("Loading expression matrix...\n")
mat <- as.matrix(read.csv(input_csv, row.names = 1, check.names = FALSE))
rownames(mat) <- make.unique(rownames(mat))
colnames(mat) <-  unlist(str_split(colnames(mat),"_",2))[seq(1,length(unlist(str_split(colnames(mat),"_",2))),2)]
cell_ids <- rownames(mat)
cat("Matrix dimensions:", nrow(mat), "cells ×", ncol(mat), "TFBS\n")

# ---------------- Load pathways ----------------
if (exists("features")) {
  families_list <- get("features")
  cat("Using 'families_list' from environment; num families:", length(families_list), "\n")
} else if (!is.null(families_rds) && file.exists(families_rds)) {
  families_list <- readRDS(families_rds)
  cat("Loaded families_list from", families_rds, " (# families:", length(families_list), ")\n")
} else {
  stop("No 'families_list' found.")
}

tfbs_universe <- colnames(mat)
families_list <- lapply(families_list, function(gs) intersect(gs, tfbs_universe)) 
families_list <- families_list[sapply(families_list, length) >= 3] 
cat("After filtering, using", length(families_list), "TFBS families (>=3 TFBS)\n")

# ---------------- Define helper functions ----------------
run_magic_impute <- function(mat_sub, seed = NULL) {
  mg <- magic(mat_sub, t = 2, seed = seed)
  as.matrix(mg$result)
  
}

compute_family_scores <- function(expr_mat, families_list) {
  tfbs_fams <- c()
  fams <- c()
  
  for (family in names(families_list)){
    TFs_temp <- families_list[[family]]
    num <- length(TFs_temp)
    if (num > 0){
      if (num > 1){
        zscores <- as.numeric(rowSums(expr_mat[,TFs_temp])/sqrt(num))
        fams <- c(fams, family)
        tfbs_fams <- rbind(tfbs_fams,zscores)
      } else if  (num == 1){
        fams <- c(fams, family)
        tfbs_fams <- rbind(tfbs_fams,expr_mat[,TFs_temp])
      }
    }
    
  }
  rownames(tfbs_fams) <- fams
  colnames(tfbs_fams) <- rownames(expr_mat)
  
  return(tfbs_fams)
}

zscore_matrix <- function(mat) {
  z <- scale(mat, center = TRUE, scale = TRUE)
  z[is.na(z)] <- 0
  z
}

cluster_hier_pathway_raw <- function(pathway_z, K = n_clusters, scale = FALSE) {
  mat <- if (scale) zscore_matrix(pathway_z) else pathway_z # decide whether or not to rescale what comes out of GSVA
  d <- dist(mat, method = "euclidean")
  hc <- fastcluster::hclust(d, method = "ward.D2")
  cl <- cutree(hc, k = K)
}

# ---------------- Full-data reference ----------------
cat("Running MAGIC on full data (reference)...\n")
mg_full <- run_magic_impute(mat, seed = seed_base)
gsva_full <- compute_family_scores(mg_full, families_list)
ref_clusters <- cluster_hier_pathway_raw(t(gsva_full), K = n_clusters, scale = FALSE)

# ---------------- Run bootstrap ----------------

n <- nrow(mat)
O <- matrix(0L, n, n, dimnames = list(cell_ids, cell_ids))
C <- matrix(0L, n, n, dimnames = list(cell_ids, cell_ids))

ARI_vec <- numeric(n_boot)

# For consensus tfbs family scores
consensus_tfbs_sum <- matrix(0, nrow = length(families_list), ncol = n,
                           dimnames = list(names(families_list),cell_ids))

counts <- matrix(0, nrow = length(families_list), ncol = n,
               dimnames = list(names(families_list),cell_ids))

pb <- txtProgressBar(min = 0, max = n_boot, style = 3)
for (b in seq_len(n_boot)) {
    setTxtProgressBar(pb, b)
    seed_b <- seed_base + b
    set.seed(seed_b)
    
    # Sample cells with replacement
    sample_idx <- sample(seq_len(n), size = n, replace = TRUE)
    sampled_ids <- cell_ids[sample_idx]
    mat_sub <- mat[sampled_ids, , drop = FALSE]
    rownames(mat_sub) <- sampled_ids
    
    
    # MAGIC
    imputed_sub <- run_magic_impute(mat_sub, seed = seed_b)
    rownames(imputed_sub) <- sub("\\.", "#", rownames(imputed_sub))
    
    
    # Average duplicates if requested
    if (avg_duplicates) {
      uniq_ids <- unique(sampled_ids)
      if (length(uniq_ids) < length(sampled_ids)) {
        agg_mat <- sapply(uniq_ids, function(id)
          colMeans(imputed_sub[grep(sprintf("%s(\\.\\d+)?$",id),rownames(imputed_sub)), , drop = FALSE]))
        imputed_for_scores <- t(agg_mat)
      } else {
        imputed_for_scores <- imputed_sub
      }
    } else {
      imputed_for_scores <- imputed_sub
    }
    
    # GSVA (zscoreParam)
    imputed_for_scores_z <- zscore_matrix(imputed_for_scores)
    gsva_res <- compute_family_scores(imputed_for_scores_z, families_list)
    pathway_z <- gsva_res 
    
    # Update consensus pathway sum
    for (i in seq_along(colnames(pathway_z))) {
      consensus_tfbs_sum[,colnames(pathway_z)[i]] <- consensus_tfbs_sum[, colnames(pathway_z)[i]] +
        pathway_z[,i]
      counts[, colnames(pathway_z)[i]] <- counts[, colnames(pathway_z)[i]] + 1
    }
    
    # Cluster
    clusters <- cluster_hier_pathway_raw(t(pathway_z), K = n_clusters, scale = FALSE)
    
    uniq_ids <- names(clusters)
    idx <- match(uniq_ids, cell_ids)
    for (i in seq_along(idx)) {
      for (j in seq_along(idx)) {
        O[idx[i], idx[j]] <- O[idx[i], idx[j]] + 1L
        if (clusters[i] == clusters[j]) C[idx[i], idx[j]] <- C[idx[i], idx[j]] + 1L
      }
    }

    # Computed the adjusted rand index to quantify cluster similarity between reference and boostrap clusters
    common <- intersect(uniq_ids, names(ref_clusters))
    if (length(common) > 1) {
      ARI_vec[b] <- adjustedRandIndex(clusters[common], ref_clusters[common])
    } else {
      ARI_vec[b] <- NA
    }

}
close(pb)
cat("\nBootstrapping complete.\n")

# ---------------- Consensus pathway scores ----------------
consensus_tfbs_scores <- consensus_tfbs_sum / counts


# ---------------- Compute co-assignment probabilities ----------------
coassign_prob <- C / pmax(O, 1)


# ---------------- Consensus clustering ----------------
dist_cons <- as.dist(1 - coassign_prob)
hc_cons <- fastcluster::hclust(dist_cons, method = "ward.D2")
consensus_clusters <- cutree(hc_cons, k = n_clusters)
names(consensus_clusters) <- cell_ids

# ---------------- Compute per-cell stability scores ----------------

min_cooccur <- 3  # set to 0 to disable thresholding

cell_stability <- sapply(cell_ids, function(cid) {
cl <- consensus_clusters[cid]
members <- names(consensus_clusters)[consensus_clusters == cl]
# exclude self
members_no_self <- setdiff(members, cid)
if (length(members_no_self) == 0) return(NA_real_)       # singleton cluster

# optionally require minimum co-occurrence count
if (min_cooccur > 0) {
  valid_members <- members_no_self[vapply(members_no_self, function(m) {
    O[cid, m] >= min_cooccur
  }, logical(1))]
} else {
  valid_members <- members_no_self
}
if (length(valid_members) == 0) return(NA_real_)         # not enough info

mean(coassign_prob[cid, valid_members], na.rm = TRUE)
})
names(cell_stability) <- cell_ids

# Cluster-level summary: mean stability per consensus cluster (ignore NA cells)
cluster_stability <- tapply(cell_stability, consensus_clusters, function(x) mean(x, na.rm = TRUE))

# ---------------- Summaries ----------------
cat("Mean cell stability:", round(mean(cell_stability, na.rm = TRUE), 3), "\n")
cat("Mean ARI vs full clustering:", round(mean(ARI_vec, na.rm = TRUE), 3), "\n")

# ---------------- Save outputs ----------------
out <- list(
params = list(n_boot = n_boot, n_clusters = n_clusters, seed = seed_base),
tfbs_list = families_list,
ARI_vec = ARI_vec,
coassign_prob = coassign_prob,
cell_stability = cell_stability,
cluster_stability = cluster_stability,
consensus_clusters = consensus_clusters,
ref_clusters = ref_clusters,
pathway_z_full = gsva_full,
consensus_tfbs_scores = consensus_tfbs_scores
)

saveRDS(out, file = sprintf("%s_Nclust_%s.rds", save_prefix, n_clusters))
write.csv(data.frame(cell_id = names(consensus_clusters),
                   consensus_cluster = consensus_clusters,
                   stability = cell_stability),
        file = sprintf("%s_Nclust_%s_consensus_clusters.csv", save_prefix, n_clusters), row.names = FALSE)
write.csv(data.frame(ARI = ARI_vec),
        file = sprintf("%s_Nclust_%s_ARI_per_bootstrap.csv", save_prefix, n_clusters), row.names = FALSE)
write.csv(consensus_tfbs_scores,
        file = sprintf("%s_Nclust_%s_consensus_tfbs_scores.csv", save_prefix, n_clusters), row.names = TRUE)
                      